# **Data and Information Quality Project**

In [ ]:
import pandas as pd
from ydata_profiling import ProfileReport
import numpy as np
import json
import os
import re

In [ ]:
DATASET = pd.read_csv('Comune-di-Milano-Servizi-alla-persona-parrucchieri-estetisti(in).csv',sep=';',encoding='unicode_escape')


In [ ]:
DATASET

# *Data Quality Assesement*


In [ ]:
#Data information
DATASET.info()

##### Single column analisys

In [ ]:
# Distinct values for each column
DATASET.nunique()

In [ ]:
# Uniqueness percentage for each column
UNIQUENESS = (DATASET.nunique() / DATASET.shape[0]) * 100
UNIQUENESS

In [ ]:
#Information about the type of esercizio
DATASET.value_counts("Tipo esercizio pa")

##### Completeness

In [ ]:
# For each column
NULL_VALUES = DATASET.isnull().sum()
NOT_NULL_VALUES = DATASET.notnull().sum()
ROWS = DATASET.shape[0]
COMPLETENESS = NOT_NULL_VALUES / ROWS
COMPLETENESS = COMPLETENESS.map('{:.2%}'.format)
COMPLETENESS

In [ ]:
# For the entire dataset
TOT_NULL_VALUES = DATASET.isnull().sum().sum()   # TODO: Controllare se abbiamo celle null con valori differenti
TOT_NOT_NULL_VALUES = DATASET.notnull().sum().sum()
TOT_COMPLETENESS = TOT_NOT_NULL_VALUES / DATASET.size
TOT_COMPLETENESS = '{:.2%}'.format(TOT_COMPLETENESS)
TOT_COMPLETENESS

##### Duplication

In [ ]:
DATASET.duplicated().any()

In [ ]:
DATASET[DATASET.duplicated()]

# *Data Profiling*


In [ ]:
#profile = ProfileReport(DATASET, title=" Report comune di Milano Servizi alla persona di parrucchieri e estetisti")
#profile.to_file("Report comune di Milano Servizi alla persona di parrucchieri e estetisti.html")

# *Data Wrangling*

Renaming and Sorting

In [ ]:
DATASET.rename(columns={"Tipo esercizio pa":"Tipo esercizio", "Prevalente":"Attivita Primaria", "ZD":"Municipio"},inplace=True)

In [ ]:
DATASET = DATASET.sort_values(by = ['Tipo esercizio', "Ubicazione"], ascending=True)
DATASET.head()

Standardization

In [ ]:
def to_upper_safe(x):
    if isinstance(x, str):
        return x.upper()
    return x

DATASET["Tipo esercizio"] = DATASET["Tipo esercizio"].map(lambda x: to_upper_safe(x) if pd.notnull(x) else x)

In [ ]:
# Transform "Tipo Esercizio"

new_cols = DATASET["Tipo esercizio"].str.split(";", expand=True)
new_cols = new_cols.apply(lambda col: col.str.strip())
new_cols = new_cols.replace("", np.nan)
new_cols = new_cols.replace({None: np.nan})
new_cols.columns = [f"Tipo_esercizio_{i+1}" for i in range(new_cols.shape[1])]

DATASET = pd.concat([DATASET, new_cols], axis=1)

split_cols = DATASET[[c for c in DATASET.columns if c.startswith("Tipo_esercizio_")]]

acconciatori = {"ACCONCIATORE", "PARRUCCHIERE MISTO", "PARRUCCHIERE PER SIGNORA", "PARRUCCHIERE PER UOMO", "BARBIERE"}
estetisti = {"ESTETISTA", "ESTETISTA IN PROFUMERIA", "TIPO A - REG.2003", "TIPO A ESTETICA MANUALE", "TIPO A-B-C-D", "MANICURE", "PEDICURE ESTETICO", "TRUCCATORE"}
centri_abbronzatura = {"CENTRO ABBRONZATURA", "TIPO B CENTRO DI ABBRONZATURA", "CENTRO BENESSERE", "CENTRO MASSAGGI", "TIPO A-B-C-D"}
trattamenti = {"TIPO C TRATT.ESTETICI DIMAGRIM", "TIPO D ESTET.APPAR.ELETTROMECC", "TIPO A-B-C-D"}
tatuaggi = {"ESECUZIONE DI TATUAGGI E PIERCING"}

def has_any_from_group(group_set):
    return split_cols.isin(group_set).any(axis=1)

DATASET["ACCONCIATORE"] = has_any_from_group(acconciatori)
DATASET["ESTETISTA"] = has_any_from_group(estetisti)
DATASET["CENTRO ABBRONZATURA"] = has_any_from_group(centri_abbronzatura)
DATASET["TRATTAMENTO"] = has_any_from_group(trattamenti)
DATASET["TATUAGGI E PIERCING"] = has_any_from_group(tatuaggi)

DATASET = DATASET.drop(columns=split_cols.columns)
DATASET
DATASET.to_csv("cleaned_tot.csv", index=True)

Column Splitting

In [ ]:
# Split "Ubicazione"
df = DATASET.copy()

# Normalization for Ubicazione
df['Ubicazione'] = df['Ubicazione'].str.replace('NUM', 'N')

# Use a different name for the DATASET for not change the original one

# Splitting the "Ubicazione" column on "N." to separate address and addition data
split_column = df['Ubicazione'].str.split(r"\bN\.", n=1, expand=True)
split_column.columns = ['Ubicazione_effettiva', 'Ubicazione_data']
# Clean the Ubicazione_data by stripping leading/trailing whitespace
split_column['Ubicazione_data'] = split_column['Ubicazione_data'].str.strip()
# Add split columns to df
df = pd.concat([df, split_column], axis=1)

# Splitting the "Ubicazione_data" column on "(" to separate civico and z.d.
split_column_2 = df['Ubicazione_data'].str.split(r"\(Z\.D\.", n=1, expand=True)
split_column_2.columns = ['Civico_to_check', 'Municipio_to_check']
# Add to original database
df = pd.concat([df, split_column_2], axis=1)
# Remove ";" from "Civico_to_check"
df['Civico_to_check'] = df['Civico_to_check'].str.replace(";", "", regex=False)
df['Civico_to_check'] = df['Civico_to_check'].str.strip()
# Remove the closing parenthesis ")" and the ZD description from the "Municipio_to_check" column
df["Municipio_to_check"] = df["Municipio_to_check"].str.replace("Z.D.", "", regex=False)
df["Municipio_to_check"] = df["Municipio_to_check"].str.replace(")", "", regex=False)
df['Municipio_to_check'] = df['Municipio_to_check'].str.strip()
# Convert null/empty values to pd.NA
df['Civico_to_check'] = df['Civico_to_check'].fillna(pd.NA)
df['Municipio_to_check'] = df['Municipio_to_check'].fillna(pd.NA)

# Splitting the "Ubicazione_effettiva" column on the first " " to separate "Tipo via" and "Via"
split_column_3 = df['Ubicazione_effettiva'].str.split(" ", n=1, expand=True)
split_column_3.columns = ['Tipo via_to_check', 'Via_to_check']
# Add to original database
df = pd.concat([df, split_column_3], axis=1)
# Convert null/empty values to pd.NA for Tipo via and Via
df['Tipo via_to_check'] = df['Tipo via_to_check'].fillna(pd.NA)
df['Tipo via_to_check'] = df['Tipo via_to_check'].str.strip()
df['Via_to_check'] = df['Via_to_check'].fillna(pd.NA)
df['Via_to_check'] = df['Via_to_check'].str.strip()

df = df.drop(['Ubicazione_effettiva', 'Ubicazione_data'], axis=1)
df[["Ubicazione", "Tipo via_to_check", "Via_to_check", "Civico_to_check", "Municipio_to_check"]]

Civico and Municipio

In [ ]:
col = "Ubicazione" 

pattern = r"""
^\s*
(?P<tipo_via_check>\S+)         # prima parola = tipo via
\s+
(?P<indirizzo_check>.*?)        # fino a prima di 'N.'
\s+N\.\s*
(?P<Civico_to_check>\S+)           # civico (numero + eventuali lettere)     
.*?Z\.D\.\s*
(?P<Municipio_to_check>\d+)        # SOLO le cifre dopo Z.D.
"""

parsed = DATASET[col].str.extract(pattern, flags=re.VERBOSE)

DATASET = pd.concat([DATASET, parsed], axis=1)
DATASET['Civico'] = DATASET['Civico'].fillna(0)
DATASET['Civico_to_check'] = DATASET['Civico_to_check'].fillna(0)
DATASET['Municipio'] = DATASET['Municipio'].fillna(0)
DATASET['Municipio_to_check'] = DATASET['Municipio_to_check'].fillna(0)
DATASET

In [ ]:
# Clean and convert Civico_to_check and ZD_to_check before comparison
def clean_numeric_string(value):
    if pd.isna(value):
        return pd.NA, pd.NA
    
    if isinstance(value, str):
        value_str = value.strip()
        if not value_str:
            return pd.NA, pd.NA
        
        # Check if value contains at least one digit, otherwise return pd.NA
        if not re.search(r'\d', value_str):
            return pd.NA, pd.NA
        
        # Find all numeric sequences with optional suffixes (like 7, 0061, 22c, 6/2)
        all_numbers = re.findall(r'\d+(?:[a-zA-Z](?![a-zA-Z])|/\d*)?', value_str)
        
        if not all_numbers:
            return pd.NA, value
        
        # Look for numbers with leading zeros (e.g., 0061, 008, 002a)
        numbers_with_leading_zeros = [num for num in all_numbers if re.match(r'^0\d+', num)]
        
        # Prioritize numbers with leading zeros, otherwise take the first number
        if numbers_with_leading_zeros:
            numeric_part = numbers_with_leading_zeros[0]
        else:
            numeric_part = all_numbers[0]
        
        # Find the position of the selected number and extract remaining text after it
        num_position = value_str.find(numeric_part)
        remaining = value_str[num_position + len(numeric_part):].strip()
        additional_part = remaining if remaining else pd.NA
        
        # Remove leading zeros but keep at least one digit, preserving suffixes like /2, a, b, etc.
        numeric_part = re.sub(r'^0+(?=\d)', '', numeric_part)
        if not numeric_part:
            numeric_part = "0"
        
        return numeric_part, additional_part
        
    elif isinstance(value, (int, float)):
        return str(int(value)), pd.NA
    
    return pd.NA, pd.NA

# # Splitting the "Ubicazione_effettiva" column on the first " " to separate "Tipo via" and "Via"
# split_column_3 = df['Ubicazione_effettiva'].str.split(" ", n=1, expand=True)
# split_column_3.columns = ['Tipo via_to_check', 'Via_to_check']
# # Add to original database
# df = pd.concat([df, split_column_3], axis=1)
# # Convert null/empty values to pd.NA for Tipo via and Via
# df['Tipo via_to_check'] = df['Tipo via_to_check'].fillna(pd.NA)
# df['Tipo via_to_check'] = df['Tipo via_to_check'].str.strip()
# df['Via_to_check'] = df['Via_to_check'].fillna(pd.NA)
# df['Via_to_check'] = df['Via_to_check'].str.strip()

# df = df.drop(['Ubicazione_effettiva', 'Ubicazione_data'], axis=1)
# df[["Ubicazione", "Tipo via_to_check", "Via_to_check", "Civico_to_check", "Municipio_to_check"]]

Tipo via

In [ ]:
# Check if there are some differences between Tipo via and Tipo via_to_check, Via and Via_to_check
condition = (
    (df['Tipo via'].fillna(' ') != df['Tipo via_to_check'].fillna(' ')) 
)
check_tipo_via = df[condition]
check_tipo_via[["Tipo via", "Tipo via_to_check"]]

Via

Codice via

# *Error Detection & Correction*

Civico and Municipio

In [ ]:
# Fix "Ubicazione"

df.loc[:, 'Civico'] = df.apply(
    lambda row: row["Civico_to_check"] if row["Civico"] == '0' and row["Civico_to_check"] != '0' else 
                (row["Civico"] if row["Civico_to_check"] == '0' else 
                (row["Civico_to_check"] if row["Civico"] != row["Civico_to_check"] else row["Civico"])), axis=1)
df.loc[:, 'Municipio'] = df.apply(
    lambda row: row["Municipio_to_check"] if row["Municipio"] == '0' and row["Municipio_to_check"] != '0' else 
                (row["Municipio"] if row["Municipio_to_check"] == '0' else 
                (row["Municipio_to_check"] if row["Municipio"] != row["Municipio_to_check"] else row["Municipio"])), axis=1)

df.loc[check_civico_municipio.index, ["Civico", "Civico_to_check", "Civico_additional", "Municipio", "Municipio_to_check", "Municipio_additional"]]

NOTA: C'è solo un Municipio a 0

Tipo via

In [ ]:
# Save in csv file all the types of Tipo Via
DATASET['Tipo via'].drop_duplicates().dropna().to_csv('tipo_via_types.csv', index=False)
valid_tipo_via = pd.read_csv('tipo_via_types.csv')['Tipo via'].tolist()

df['Tipo via'] = df['Tipo via'].astype(str).replace('<NA>', ' ').replace('nan', ' ')
df['Tipo via_to_check'] = df['Tipo via_to_check'].astype(str).replace('<NA>', ' ').replace('nan', ' ')
df['Tipo via_to_check'] = df['Tipo via_to_check'].apply(
    lambda x: x if x in valid_tipo_via else ' '
)

df.loc[:, 'Tipo via'] = df.apply(
    lambda row: row['Tipo via_to_check'] if row['Tipo via_to_check'] != ' ' and row['Tipo via'] != row['Tipo via_to_check'] else row['Tipo via'],
    axis=1
)

df.loc[check_tipo_via.index, ["Tipo via", "Tipo via_to_check"]]


Via

Codice via

In [ ]:
# # Clean and convert Civico_to_check and ZD_to_check before comparison
# def clean_numeric_string(value):
#     if pd.isna(value):
#         return pd.NA, pd.NA
    
#     if isinstance(value, str):
#         value_str = value.strip()
#         if not value_str:
#             return pd.NA, pd.NA
        
#         # Check if value contains at least one digit, otherwise return pd.NA
#         if not re.search(r'\d', value_str):
#             return pd.NA, pd.NA
        
#         # Find all numeric sequences with optional suffixes (like 7, 0061, 22c, 6/2)
#         all_numbers = re.findall(r'\d+(?:[a-zA-Z](?![a-zA-Z])|/\d*)?', value_str)
        
#         if not all_numbers:
#             return pd.NA, value
        
#         # Look for numbers with leading zeros (e.g., 0061, 008, 002a)
#         numbers_with_leading_zeros = [num for num in all_numbers if re.match(r'^0\d+', num)]
        
#         # Prioritize numbers with leading zeros, otherwise take the first number
#         if numbers_with_leading_zeros:
#             numeric_part = numbers_with_leading_zeros[0]
#         else:
#             numeric_part = all_numbers[0]
        
#         # Find the position of the selected number and extract remaining text after it
#         num_position = value_str.find(numeric_part)
#         remaining = value_str[num_position + len(numeric_part):].strip()
#         additional_part = remaining if remaining else pd.NA
        
#         # Remove leading zeros but keep at least one digit, preserving suffixes like /2, a, b, etc.
#         numeric_part = re.sub(r'^0+(?=\d)', '', numeric_part)
#         if not numeric_part:
#             numeric_part = "0"
        
#         return numeric_part, additional_part
        
#     elif isinstance(value, (int, float)):
#         return str(int(value)), pd.NA
    
#     return pd.NA, pd.NA

# # Apply cleaning to Civico_to_check and save both parts
# df[['Civico_to_check', 'Civico_additional']] = df['Civico_to_check'].apply(
#     lambda x: pd.Series(clean_numeric_string(x))
# )
# df[['Municipio_to_check', 'Municipio_additional']] = df['Municipio_to_check'].apply(
#     lambda x: pd.Series(clean_numeric_string(x))
# )

# # Convert to string for comparison, replacing pd.NA with empty string
# df['Civico'] = df['Civico'].astype(str).replace('<NA>', '0').replace('nan', '0')
# df['Municipio'] = df['Municipio'].astype(str).replace('<NA>', '0').replace('nan', '0')
# df['Civico_to_check'] = df['Civico_to_check'].astype(str).replace('<NA>', '0')
# df['Municipio_to_check'] = df['Municipio_to_check'].astype(str).replace('<NA>', '0')
# # Check the not right values
# condition = ((df['Civico_to_check'] != df['Civico']) | (df['Municipio_to_check'] != df['Municipio']))
# check_civico_municipio = df[condition][['Civico', 'Civico_to_check', 'Civico_additional', 'Municipio', 'Municipio_to_check', 'Municipio_additional']]
# check_civico_municipio

# *Error Detection & Correction*

In [ ]:
# Drop rows where "Attivita Primaria" and "Tipo esercizio" are null
DATASET = DATASET.dropna(subset=["Attivita Primaria", "Tipo esercizio"], how='all')

In [ ]:
# If "Attivita Primaria" is null, fill with name of the first True column of "Tipo esercizio"

# Maybe review this part --> maybe we are generalizing too much and losing information

tipo_bool_cols = ["ACCONCIATORE", "ESTETISTA", "CENTRO ABBRONZATURA", "TRATTAMENTO", "TATUAGGI E PIERCING"]
tipo_bool = DATASET[tipo_bool_cols]

has_true = tipo_bool.any(axis=1)

first_true_col = tipo_bool.idxmax(axis=1)

first_true_col = first_true_col.where(has_true, np.nan)

mask_att_null = DATASET["Attivita Primaria"].isna()

DATASET.loc[mask_att_null, "Attivita Primaria"] = first_true_col[mask_att_null]

DATASET = DATASET.drop(columns='Tipo esercizio')

In [ ]:
# Fix "Ubicazione"

# Fix Civico_to_check and Municipio_to_check
DATASET.loc[:, 'Civico'] = DATASET.apply(
    lambda row: row["Civico_to_check"] if row["Civico"] == 0 and row["Civico_to_check"] != 0 else 
                (row["Civico"] if row["Civico_to_check"] == 0 else 
                (row["Civico_to_check"] if row["Civico"] != row["Civico_to_check"] else row["Civico"])), axis=1)
DATASET.loc[:, 'Municipio'] = DATASET.apply(
    lambda row: row["Municipio_to_check"] if row["Municipio"] == 0 and row["Municipio_to_check"] != 0 else 
                (row["Municipio"] if row["Municipio_to_check"] == 0 else 
                (row["Municipio_to_check"] if row["Municipio"] != row["Municipio_to_check"] else row["Municipio"])), axis=1)

DATASET

In [ ]:
# Maybe other fixes

In [ ]:
# *Null Values Handling*

In [ ]:
# Drop rows where "Attivita Primaria" and "Tipo esercizio" are null
DATASET = DATASET.dropna(subset=["Attivita Primaria", "Tipo esercizio"], how='all')

In [ ]:
# If "Attivita Primaria" is null, fill with name of the first True column of "Tipo esercizio"

# Maybe review this part --> maybe we are generalizing too much and losing information

tipo_bool_cols = ["ACCONCIATORE", "ESTETISTA", "CENTRO ABBRONZATURA", "TRATTAMENTO", "TATUAGGI E PIERCING"]
tipo_bool = DATASET[tipo_bool_cols]

has_true = tipo_bool.any(axis=1)

first_true_col = tipo_bool.idxmax(axis=1)

first_true_col = first_true_col.where(has_true, np.nan)

mask_att_null = DATASET["Attivita Primaria"].isna()

DATASET.loc[mask_att_null, "Attivita Primaria"] = first_true_col[mask_att_null]

DATASET = DATASET.drop(columns='Tipo esercizio')

In [ ]:
DATASET

In [ ]:
# Fill "Superficie lavorativa" with median value grouping by "Tipo esercizio"
group_median = (
    DATASET
    .groupby(tipo_bool_cols)["Superficie lavorativa"]
    .transform("median")   # median is computed ignoring NaN
)

mask_superficie_null = DATASET["Superficie lavorativa"].isna()
DATASET.loc[mask_superficie_null, "Superficie lavorativa"] = group_median[mask_superficie_null]

In [ ]:
# Fill "Superficie altri usi" with 0
DATASET["Superficie altri usi"] = DATASET["Superficie altri usi"].fillna(0)

In [ ]:
# Fill "Codice via" with values extracted from "Ubicazione" if existing, 0 otherwise

In [ ]:
# Fill "Municipio" with values extracted from "Ubicazione" if existing, 0 otherwise

# *Outlier Detection*

In [ ]:
# Compute Z-score on certain columns and trop outliers

# *Duplicate Detection*

In [ ]:
# Drop exact duplicates

In [ ]:
# Use 'Sorted Neighbourhood' to find possible duplicates

In [ ]:
# Drop new found possible duplicates